In [140]:
import pandas as pd
import numpy as np
import pycountry

In [141]:
#Importation des fichiers
df_elec = pd.read_csv(
    "access_elec.csv", 
    quotechar='"', 
    skipinitialspace=True,
    na_values=['']
)
df_lpi = pd.read_csv(
    "lpi.csv", 
    quotechar='"', 
    skipinitialspace=True,
    na_values=['']
)
df_politic = pd.read_csv("political-polarization-score.csv")
df_idh = pd.read_csv("idh.csv")
df_dist = pd.read_csv("dist_fra.csv", sep=";")
df_alim_saine = pd.read_csv("cout_alim_saine.csv")
df_population = pd.read_csv("population.csv")
df_pib = pd.read_csv("pib.csv")
df_tarifs = pd.read_csv("tariffs.csv")
df_consommation_poulet = pd.read_csv("nourriture.csv")
df_importation = pd.read_csv("import_poulet.csv")


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_elec</h2>
</div>

In [142]:
# 1. Sélectionner uniquement les colonnes d'années (dans l'ordre chronologique)
col_annees = [col for col in df_elec.columns if col.isdigit()]
col_annees.sort() # S'assurer que l'ordre va bien de 1960 à 2025

# 2. Appliquer le Forward Fill ligne par ligne (axis=1) sur les colonnes d'années
# Cela va copier la dernière valeur non-nulle vers la droite pour chaque pays
df_elec[col_annees] = df_elec[col_annees].ffill(axis=1)

# 3. Maintenant, tu peux faire ton melt de manière classique
id_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']

df_elec_melt = pd.melt(
    df_elec, 
    id_vars=id_cols, 
    value_vars=col_annees,
    var_name='Year', 
    value_name='Access_Elec_Pct'
)

df_elec_melt['Year'] = df_elec_melt['Year'].astype(int)

In [143]:
# 1. On isole uniquement les colonnes qui sont des années
col_annees = [col for col in df_elec.columns if col.isdigit()]

# 2. On compte le nombre de valeurs non nulles pour chaque année
valeurs_par_annee = df_elec[col_annees].notna().sum()

# 3. On trouve l'année qui a le maximum de valeurs
meilleure_annee = valeurs_par_annee.idxmax()
max_valeurs = valeurs_par_annee.max()

print(f"🏆 L'année la plus complète est {meilleure_annee} avec {max_valeurs} pays renseignés.\n")

# 4. On affiche le Top 5 des années les plus exploitables pour ton projet
print("📊 Top 5 des années les plus complètes :")
print(valeurs_par_annee.sort_values(ascending=False).head(20))

🏆 L'année la plus complète est 2009 avec 264 pays renseignés.

📊 Top 5 des années les plus complètes :
2011    264
2010    264
2009    264
2019    264
2012    264
2013    264
2014    264
2015    264
2016    264
2017    264
2018    264
2023    264
2020    264
2021    264
2022    264
2025    264
2024    264
2007    263
2008    263
2003    261
dtype: int64


In [144]:
df_elec = df_elec_melt[['Country Name', 'Country Code', 'Year', 'Access_Elec_Pct']]

In [145]:
df_elec

,Country Name,Country Code,Year,Access_Elec_Pct
0,Aruba,ABW,1960,NaN
1,Africa Eastern and Southern,AFE,1960,NaN
2,Afghanistan,AFG,1960,NaN
3,Africa Western and Central,AFW,1960,NaN
4,Angola,AGO,1960,NaN
...,...,...,...,...
17551,Kosovo,XKX,2025,100.0
17552,"Yemen, Rep.",YEM,2025,83.6
17553,South Africa,ZAF,2025,87.7
17554,Zambia,ZMB,2025,51.1


In [146]:
# 1. On passe le tableau en format "large" temporairement pour avoir les années en colonnes
df_elec_pivot = df_elec.pivot_table(
    index='Country Name', 
    columns='Year', 
    values='Access_Elec_Pct', 
    aggfunc='mean' # 'mean' ou 'sum' (ici mean est plus sûr pour des pourcentages s'il y a des doublons)
)

# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_elec_pivot['%_evo_elec'] = (df_elec_pivot[2023] - df_elec_pivot[2013]) / df_elec_pivot[2013]

# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_elec = df_elec_pivot[['%_evo_elec']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_elec = pd.merge(df_elec, df_final_elec, on='Country Name', how='left')
df_elec = df_elec[df_elec['Year'] == 2023]

In [147]:
df_elec = df_elec.rename(columns={'Country Name' : 'Zone', 'Country Code' : 'Code', 'Year' : 'Année'})
df_elec

,Zone,Code,Année,Access_Elec_Pct,%_evo_elec
16758,Aruba,ABW,2023,100.000000,0.000000
16759,Africa Eastern and Southern,AFE,2023,50.667516,0.597157
16760,Afghanistan,AFG,2023,85.300000,0.254412
16761,Africa Western and Central,AFW,2023,57.069267,0.209235
16762,Angola,AGO,2023,51.100000,0.330729
...,...,...,...,...,...
17019,Kosovo,XKX,2023,100.000000,0.000000
17020,"Yemen, Rep.",YEM,2023,83.600000,0.105820
17021,South Africa,ZAF,2023,87.700000,0.029343
17022,Zambia,ZMB,2023,51.100000,0.697674


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_lpi</h2>
</div>

In [148]:
# 1. Sélectionner uniquement les colonnes d'années (dans l'ordre chronologique)
col_annees = [col for col in df_lpi.columns if col.isdigit()]
col_annees.sort() # S'assurer que l'ordre va bien de 1960 à 2025

# 2. Appliquer le Forward Fill ligne par ligne (axis=1) sur les colonnes d'années
# Cela va copier la dernière valeur non-nulle vers la droite pour chaque pays
df_lpi[col_annees] = df_lpi[col_annees].ffill(axis=1)

# 3. Maintenant, tu peux faire ton melt de manière classique
id_cols = ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code']

df_lpi_melt = pd.melt(
    df_lpi, 
    id_vars=id_cols, 
    value_vars=col_annees,
    var_name='Year', 
    value_name='LPI'
)

df_lpi_melt['Year'] = df_lpi_melt['Year'].astype(int)

# 1. On isole uniquement les colonnes qui sont des années
col_annees = [col for col in df_lpi.columns if col.isdigit()]

# 2. On compte le nombre de valeurs non nulles pour chaque année
valeurs_par_annee = df_lpi[col_annees].notna().sum()

# 3. On trouve l'année qui a le maximum de valeurs
meilleure_annee = valeurs_par_annee.idxmax()
max_valeurs = valeurs_par_annee.max()

print(f"🏆 L'année la plus complète est {meilleure_annee} avec {max_valeurs} pays renseignés.\n")

# 4. On affiche le Top 5 des années les plus exploitables pour ton projet
print("📊 Top 5 des années les plus complètes :")
print(valeurs_par_annee.sort_values(ascending=False).head(20))

🏆 L'année la plus complète est 2022 avec 217 pays renseignés.

📊 Top 5 des années les plus complètes :
2023    217
2022    217
2024    217
2025    217
2019    215
2018    215
2016    215
2017    215
2020    215
2021    215
2015    213
2014    213
2010    211
2011    211
2012    211
2013    211
2008    197
2009    197
2007    197
1960      0
dtype: int64


In [149]:
df_lpi = df_lpi_melt[['Country Name', 'Country Code', 'Year', 'LPI']]

# 1. On passe le tableau en format "large" temporairement pour avoir les années en colonnes
df_lpi_pivot = df_lpi.pivot_table(
    index='Country Name', 
    columns='Year', 
    values='LPI', 
    aggfunc='mean' # 'mean' ou 'sum' (ici mean est plus sûr pour des pourcentages s'il y a des doublons)
)
# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_lpi_pivot['%_evo_lpi'] = 100 * (df_lpi_pivot[2023] - df_lpi_pivot[2013]) / df_lpi_pivot[2013]

# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_lpi = df_lpi_pivot[['%_evo_lpi']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_lpi = pd.merge(df_lpi, df_final_lpi, on='Country Name', how='left')
df_lpi = df_lpi[df_lpi['Year'] == 2023]

df_lpi = df_lpi.rename(columns={'Country Name' : 'Zone', 'Country Code' : 'Code', 'Year' : 'Année'})
df_lpi

,Zone,Code,Année,LPI,%_evo_lpi
16758,Aruba,ABW,2023,NaN,NaN
16759,Africa Eastern and Southern,AFE,2023,2.618182,6.248301
16760,Afghanistan,AFG,2023,1.900000,-17.391304
16761,Africa Western and Central,AFW,2023,2.473333,1.021103
16762,Angola,AGO,2023,2.100000,-7.894737
...,...,...,...,...,...
17019,Kosovo,XKX,2023,NaN,NaN
17020,"Yemen, Rep.",YEM,2023,2.200000,-23.875433
17021,South Africa,ZAF,2023,3.700000,0.817439
17022,Zambia,ZMB,2023,2.530000,10.964912


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_politic</h2>
</div>

In [150]:
df_politic.head()

,Entity,Code,Year,Political polarization score
0,Afghanistan,AFG,1992,2.779
1,Afghanistan,AFG,1993,2.779
2,Afghanistan,AFG,1994,2.779
3,Afghanistan,AFG,1995,2.779
4,Afghanistan,AFG,1996,2.779


In [151]:
test = df_politic.groupby('Entity')['Political polarization score'].count().reset_index()
test.sort_values(by='Political polarization score', ascending=True)

,Entity,Political polarization score
78,Iceland,26
91,Kosovo,27
45,Democratic Republic of Vietnam,31
0,Afghanistan,34
133,Palestine/Gaza,38
...,...,...
191,World (population-weighted),126
190,World,126
184,Uruguay,126
195,Zanzibar,126


In [152]:
# 1. On passe le tableau en format "large" temporairement pour avoir les années en colonnes
df_politic_pivot = df_politic.pivot_table(
    index='Entity', 
    columns='Year', 
    values='Political polarization score'
)
# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_politic_pivot['%_evo_politic'] = 100 * (df_politic_pivot[2023] - df_politic_pivot[2013]) / df_politic_pivot[2013]

# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_politic = df_politic_pivot[['%_evo_politic']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_politic = pd.merge(df_politic, df_final_politic, on='Entity', how='left')
df_politic = df_politic[df_politic['Year'] == 2023]

df_politic = df_politic.rename(columns={'Entity' : 'Zone', 'Year' : 'Année'})
df_politic

,Zone,Code,Année,Political polarization score,%_evo_politic
31,Afghanistan,AFG,2023,2.485000,1970.833333
157,Africa,OWID_AFR,2023,0.252143,68.055225
279,Africa (population-weighted),NaN,2023,0.088952,-53.556728
393,Albania,ALB,2023,-0.622000,-59.319817
519,Algeria,DZA,2023,-1.563000,19.131098
...,...,...,...,...,...
22372,World (population-weighted),NaN,2023,0.838582,-1117.730261
22425,Yemen,YEM,2023,2.850000,67.844523
22613,Zambia,ZMB,2023,-0.521000,-2184.000000
22739,Zanzibar,OWID_ZAN,2023,-1.348000,-15.220126


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_idh</h2>
</div>

In [153]:
df_idh

,Entity,Code,Year,Human Development Index,World region according to OWID
0,Afghanistan,AFG,1990,0.285,Asia
1,Afghanistan,AFG,1991,0.291,Asia
2,Afghanistan,AFG,1992,0.301,Asia
3,Afghanistan,AFG,1993,0.311,Asia
4,Afghanistan,AFG,1994,0.305,Asia
...,...,...,...,...,...
6599,Zimbabwe,ZWE,2019,0.584,Africa
6600,Zimbabwe,ZWE,2020,0.582,Africa
6601,Zimbabwe,ZWE,2021,0.581,Africa
6602,Zimbabwe,ZWE,2022,0.594,Africa


In [154]:
df_idh = df_idh[['Entity', 'Code', 'Year', 'Human Development Index']]
df_idh = df_idh.rename(columns={'Entity' : 'Zone', 'Year' : 'Année'})

In [155]:
test_idh = df_idh.groupby('Zone')['Human Development Index'].count().reset_index()

In [156]:
test_idh.describe()

,Human Development Index
count,215.000000
mean,30.716279
std,5.883484
min,2.000000
25%,29.000000
50%,34.000000
75%,34.000000
max,34.000000


In [157]:
test_idh[test_idh['Human Development Index'] < 10]

,Zone,Human Development Index
175,Somalia,2


In [158]:
# 1. On passe le tableau en format "large" temporairement pour avoir les années en colonnes
df_idh_pivot = df_idh.pivot_table(
    index='Zone', 
    columns='Année', 
    values='Human Development Index'
)
# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_idh_pivot['%_evo_idh'] = 100 * (df_idh_pivot[2023] - df_idh_pivot[2013]) / df_idh_pivot[2013]

# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_idh = df_idh_pivot[['%_evo_idh']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_idh = pd.merge(df_idh, df_final_idh, on='Zone', how='left')
df_idh = df_idh[df_idh['Année'] == 2023]

df_idh

,Zone,Code,Année,Human Development Index,%_evo_idh
33,Afghanistan,AFG,2023,0.496000,0.813008
57,Africa,OWID_AFR,2023,0.576059,8.436722
91,Albania,ALB,2023,0.810000,2.143758
125,Algeria,DZA,2023,0.763000,4.951857
149,Andorra,AND,2023,0.913000,5.916473
...,...,...,...,...,...
6467,Vietnam,VNM,2023,0.766000,8.806818
6501,World,OWID_WRL,2023,0.756000,4.854369
6535,Yemen,YEM,2023,0.470000,-7.297830
6569,Zambia,ZMB,2023,0.595000,6.822262


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_alim_saine</h2>
</div>

In [159]:
df_cout_alim_saine = df_alim_saine[df_alim_saine['Produit'] == 'Cost of animal source foods, PPP dollar per person per day']
df_cout_alim_saine = df_cout_alim_saine[['Zone', 'Année', 'Valeur']]
df_cout_alim_saine = df_cout_alim_saine.rename(columns={'Valeur' : 'cout_alim_saine'})

In [160]:
df_cout_alim_saine['cout_alim_saine'] = pd.to_numeric(
    df_cout_alim_saine['cout_alim_saine'])

In [161]:
df_cout_alim_saine.info()

<class 'pandas.core.frame.DataFrame'>
Index: 175 entries, 1 to 1741
Data columns (total 3 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Zone             175 non-null    object 
 1   Année            175 non-null    int64  
 2   cout_alim_saine  166 non-null    float64
dtypes: float64(1), int64(1), object(1)
memory usage: 5.5+ KB


In [162]:
df_pop_alim_saine = df_alim_saine[df_alim_saine['Produit'] == 'Number of people unable to afford a healthy diet (NUA), million']


In [163]:
df_pop_alim_saine['Année'].unique()

array([2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024])

In [164]:
df_pop_alim_saine = df_pop_alim_saine[['Zone', 'Année', 'Valeur']]
df_pop_alim_saine = df_pop_alim_saine.rename(columns={'Valeur' : 'pop_unable_to_eat_healthy'})
df_pop_alim_saine['pop_unable_to_eat_healthy'] = pd.to_numeric(
    df_pop_alim_saine['pop_unable_to_eat_healthy'], 
    errors='coerce'
)

In [165]:
df_pop_alim_saine.loc[:,'pop_unable_to_eat_healthy'] *= 1_000_000

In [166]:
df_pop_alim_saine.info()

<class 'pandas.core.frame.DataFrame'>
Index: 1400 entries, 2 to 1749
Data columns (total 3 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Zone                       1400 non-null   object 
 1   Année                      1400 non-null   int64  
 2   pop_unable_to_eat_healthy  1122 non-null   float64
dtypes: float64(1), int64(1), object(1)
memory usage: 43.8+ KB


In [167]:
# 1. On fusionne temporairement pour amener la colonne de remplacement (on la nomme 'pop_secours')
df_pop_alim_saine = pd.merge(
    df_pop_alim_saine,
    df_population[['Zone', 'Année', 'Valeur']],
    on=['Zone', 'Année'],
    how='left'
).rename(columns={'Valeur': 'pop_secours'})

# 2. On remplit les NaN de notre colonne principale avec les valeurs de secours
df_pop_alim_saine['pop_unable_to_eat_healthy'] = df_pop_alim_saine['pop_unable_to_eat_healthy'].fillna(df_pop_alim_saine['pop_secours'])
df_pop_alim_saine = df_pop_alim_saine.drop('pop_secours', axis=1)

In [168]:
# 1. On passe le tableau en format "large" temporairement pour avoir les années en colonnes
df_pop_alim_saine_pivot = df_pop_alim_saine.pivot_table(
    index='Zone', 
    columns='Année', 
    values='pop_unable_to_eat_healthy'
)
# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_pop_alim_saine_pivot['%_evo_pop_unable_to_eat_healthy'] = 100*(df_pop_alim_saine_pivot[2023] - df_pop_alim_saine_pivot[2017]) / df_pop_alim_saine_pivot[2017]
# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_pop_alim_saine = df_pop_alim_saine_pivot[['%_evo_pop_unable_to_eat_healthy']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_pop_alim_saine = pd.merge(df_pop_alim_saine, df_final_pop_alim_saine, on='Zone', how='left')
df_pop_alim_saine = df_pop_alim_saine[df_pop_alim_saine['Année'] == 2023]
df_pop_alim_saine

,Zone,Année,pop_unable_to_eat_healthy,%_evo_pop_unable_to_eat_healthy
6,Afrique du Sud,2023,3.900000e+07,11.428571
14,Albanie,2023,3.000000e+05,-57.142857
22,Algérie,2023,9.700000e+06,24.358974
30,Allemagne,2023,1.700000e+06,-26.086957
38,Angola,2023,2.600000e+07,41.304348
...,...,...,...,...
1366,Türkiye,2023,6.400000e+06,-34.020619
1374,Uruguay,2023,1.200000e+06,20.000000
1382,Viet Nam,2023,9.500000e+06,-17.391304
1390,Zambie,2023,1.700000e+07,25.000000


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_dist</h2>
</div>

In [169]:
df_dist = df_dist[['iso_d', 'dist']]
df_dist = df_dist.rename(columns={'iso_d' : 'Code'})
df_dist['dist'] = df_dist['dist'].str.replace(',', '.').astype(float)
df_dist.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 224 entries, 0 to 223
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Code    224 non-null    object 
 1   dist    224 non-null    float64
dtypes: float64(1), object(1)
memory usage: 3.6+ KB


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_population</h2>
</div>

In [170]:
df_population

,Domaine,Zone,Élément,Produit,Année,Unité,Valeur
0,Séries temporelles annuelles,Afghanistan,Population totale,Population-Estimations,1950,1000 No,7776.176
1,Séries temporelles annuelles,Afghanistan,Population totale,Population-Estimations,1951,1000 No,7879.339
2,Séries temporelles annuelles,Afghanistan,Population totale,Population-Estimations,1952,1000 No,7987.783
3,Séries temporelles annuelles,Afghanistan,Population totale,Population-Estimations,1953,1000 No,8096.698
4,Séries temporelles annuelles,Afghanistan,Population totale,Population-Estimations,1954,1000 No,8207.950
...,...,...,...,...,...,...,...
15934,Séries temporelles annuelles,Zimbabwe,Population totale,Population-Estimations,2019,1000 No,15271.368
15935,Séries temporelles annuelles,Zimbabwe,Population totale,Population-Estimations,2020,1000 No,15526.888
15936,Séries temporelles annuelles,Zimbabwe,Population totale,Population-Estimations,2021,1000 No,15797.210
15937,Séries temporelles annuelles,Zimbabwe,Population totale,Population-Estimations,2022,1000 No,16069.056


In [171]:
df_population = df_population[['Zone', 'Année', 'Valeur']]
df_population.loc[:, 'Valeur'] *= 1000
df_population = df_population.rename(columns={'Valeur':'population'})

In [172]:
# 1. On crée un tableau temporaire avec un pays par ligne et les années en colonnes
df_pivot = df_population.pivot(index='Zone', columns='Année', values='population')
# 2. On calcule l'évolution directement (ici multiplication par 100 pour l'avoir en %)
df_pivot['evo_pop'] = ((df_pivot[2023] - df_pivot[2013]) / df_pivot[2013]) * 100

# 3. On remet à plat pour obtenir un DataFrame propre avec le pays et son évolution
df_final_pop = df_pivot[['evo_pop']].reset_index()

df_population = pd.merge(df_population, df_final_pop, on='Zone', how='left')
df_population = df_population[df_population['Année'] == 2023]

In [173]:
df_population

,Zone,Année,population,evo_pop
73,Afghanistan,2023,41454761.0,31.091766
147,Afrique du Sud,2023,63212384.0,15.606770
221,Albanie,2023,2811655.0,-3.298903
295,Algérie,2023,46164219.0,20.174971
369,Allemagne,2023,84548231.0,4.326975
...,...,...,...,...
15642,Venezuela (République bolivarienne du),2023,28300854.0,-5.386913
15716,Viet Nam,2023,100352192.0,10.796901
15790,Yémen,2023,39390799.0,34.380185
15864,Zambie,2023,20723965.0,34.579966


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_pib</h2>
</div>

In [174]:
df_pib =  df_pib[['Zone', 'Année', 'Valeur']]
df_pib = df_pib.rename(columns={'Valeur':'PIB'})

In [175]:
df_pib_pivot = df_pib.pivot_table(
    index='Zone', 
    columns='Année', 
    values='PIB')

# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_pib_pivot['%_evo_pib'] = (df_pib_pivot[2023] - df_pib_pivot[2013]) * 100 / df_pib_pivot[2013]

# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_pib = df_pib_pivot[['%_evo_pib']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_pib = pd.merge(df_pib, df_final_pib, on='Zone', how='left')
df_pib = df_pib[df_pib['Année'] == 2023]

In [176]:
df_pib

,Zone,Année,PIB,%_evo_pib
53,Afghanistan,2023,1.641675e+04,-17.522051
108,Afrique du Sud,2023,3.777816e+05,-5.715917
163,Albanie,2023,2.297767e+04,79.847536
218,Algérie,2023,2.476262e+05,18.054948
273,Allemagne,2023,4.525704e+06,18.844573
...,...,...,...,...
10808,Venezuela (République bolivarienne du),2023,1.393949e+05,-62.461481
10863,Viet Nam,2023,4.297170e+05,101.075925
10899,Yémen,2023,8.757940e+03,-74.801130
10992,Zambie,2023,2.757796e+04,3.745518


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_tarifs</h2>
</div>

In [177]:
df_tarifs.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3806 entries, 0 to 3805
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype  
---  ------                                                 --------------  -----  
 0   Entity                                                 3806 non-null   object 
 1   Code                                                   3806 non-null   object 
 2   Year                                                   3806 non-null   int64  
 3   Tariff rate, applied, weighted mean, all products (%)  3806 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 119.1+ KB


In [178]:
df_tarifs = df_tarifs.rename(columns={'Entity' : 'Zone', 'Year' : 'Année', 'Tariff rate, applied, weighted mean, all products (%)' : 'tarifs'})

In [179]:
df_tarifs_pivot = df_tarifs.pivot_table(
    index='Zone', 
    columns='Année', 
    values='tarifs')

# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_tarifs_pivot['%_evo_tarifs'] = (df_tarifs_pivot[2022] - df_tarifs_pivot[2012]) * 100 / df_tarifs_pivot[2012]

# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_tarifs = df_tarifs_pivot[['%_evo_tarifs']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_tarifs = pd.merge(df_tarifs, df_final_tarifs, on='Zone', how='left')
df_tarifs = df_tarifs[df_tarifs['Année'] == 2022]

In [180]:
df_tarifs['%_evo_tarifs'] = df_tarifs['%_evo_tarifs'].fillna(0)

In [181]:
df_tarifs

,Zone,Code,Année,tarifs,%_evo_tarifs
30,Albania,ALB,2022,0.26,-76.363636
50,Algeria,DZA,2022,9.25,0.000000
68,Angola,AGO,2022,8.85,16.294350
122,Argentina,ARG,2022,6.45,7.142857
179,Australia,AUS,2022,0.99,-45.303867
...,...,...,...,...,...
3684,Uruguay,URY,2022,4.85,15.476190
3709,Vanuatu,VUT,2022,11.48,107.594937
3737,Venezuela,VEN,2022,12.84,49.476135
3760,Vietnam,VNM,2022,1.07,-70.684932


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_nourriture</h2>
</div>

In [182]:
df_consommation_poulet.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2425 entries, 0 to 2424
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Domaine  2425 non-null   object 
 1   Zone     2425 non-null   object 
 2   Élément  2425 non-null   object 
 3   Produit  2425 non-null   object 
 4   Année    2425 non-null   int64  
 5   Unité    2425 non-null   object 
 6   Valeur   2425 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 132.7+ KB


In [183]:
df_consommation_poulet = df_consommation_poulet[['Zone', 'Année', 'Valeur']]
df_consommation_poulet = df_consommation_poulet.rename(columns={'Valeur' : 'consommation_poulet'})
df_consommation_poulet

,Zone,Année,consommation_poulet
0,Afghanistan,2010,64538.38
1,Afghanistan,2011,56121.71
2,Afghanistan,2012,62405.56
3,Afghanistan,2013,66510.55
4,Afghanistan,2014,68235.20
...,...,...,...
2420,Zimbabwe,2019,115697.12
2421,Zimbabwe,2020,112834.10
2422,Zimbabwe,2021,117071.36
2423,Zimbabwe,2022,123349.73


In [184]:
df_consommation_poulet_pivot = df_consommation_poulet.pivot_table(
    index='Zone', 
    columns='Année', 
    values='consommation_poulet')

# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_consommation_poulet_pivot['%_evo_consommation_poulet'] = (df_consommation_poulet_pivot[2023] - df_consommation_poulet_pivot[2013]) * 100 / df_consommation_poulet_pivot[2013]

# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_conso_poulet = df_consommation_poulet_pivot[['%_evo_consommation_poulet']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_consommation_poulet = pd.merge(df_consommation_poulet, df_final_conso_poulet, on='Zone', how='left')
df_consommation_poulet = df_consommation_poulet[df_consommation_poulet['Année'] == 2023]
df_consommation_poulet

,Zone,Année,consommation_poulet,%_evo_consommation_poulet
13,Afghanistan,2023,32936.83,-50.478789
27,Afrique du Sud,2023,2122618.28,9.182679
41,Albanie,2023,64659.76,66.593049
55,Algérie,2023,421134.48,61.759281
69,Allemagne,2023,940232.90,5.574989
...,...,...,...,...
2368,Venezuela (République bolivarienne du),2023,500040.95,-58.129220
2382,Viet Nam,2023,1775301.76,65.044709
2396,Yémen,2023,320588.64,17.018101
2410,Zambie,2023,69726.76,46.771549


<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">df_import</h2>
</div>

In [185]:
df_importation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2365 entries, 0 to 2364
Data columns (total 7 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   Domaine  2365 non-null   object 
 1   Zone     2365 non-null   object 
 2   Élément  2365 non-null   object 
 3   Produit  2365 non-null   object 
 4   Année    2365 non-null   int64  
 5   Unité    2365 non-null   object 
 6   Valeur   2363 non-null   float64
dtypes: float64(1), int64(1), object(5)
memory usage: 129.5+ KB


In [186]:
df_importation = df_importation[['Zone', 'Année', 'Valeur']]
df_importation = df_importation.rename(columns={'Valeur' : 'importation'})

In [187]:
df_importation_pivot = df_importation.pivot_table(
    index='Zone', 
    columns='Année', 
    values='importation')

# 2. On calcule l'évolution (différence en points de pourcentage ou taux d'évolution)
df_importation_pivot['%_evo_importation'] = (df_importation_pivot[2023] - df_importation_pivot[2013]) * 100 / df_importation_pivot[2013]

# 3. On remet à plat pour obtenir le DataFrame final [Country Name, evo_elec]
df_final_import = df_importation_pivot[['%_evo_importation']].reset_index()

# 4. On fusionne l'info dans ton df_elec d'origine
df_importation = pd.merge(df_importation, df_final_import, on='Zone', how='left')
df_importation = df_importation[df_importation['Année'] == 2023]
df_importation

,Zone,Année,importation,%_evo_importation
13,Afghanistan,2023,4811.26,-90.057120
27,Afrique du Sud,2023,312852.14,-2.137985
41,Albanie,2023,47933.76,119.748590
52,Algérie,2023,28.06,inf
66,Allemagne,2023,492941.64,43.314486
...,...,...,...,...
2308,Venezuela (République bolivarienne du),2023,192.52,-99.930963
2322,Viet Nam,2023,236321.76,-52.735648
2336,Yémen,2023,116791.59,8.306833
2350,Zambie,2023,22622.43,1386.362024


In [188]:
df_importation[df_importation['%_evo_importation'] == -np.inf]

,Zone,Année,importation,%_evo_importation


In [189]:
df_importation[df_importation['%_evo_importation'] == np.inf]

,Zone,Année,importation,%_evo_importation
52,Algérie,2023,28.06,inf
211,Bangladesh,2023,47.06,inf
679,Équateur,2023,42.61,inf
1019,Inde,2023,25.02,inf
1272,Libye,2023,96518.78,inf
1851,République démocratique populaire lao,2023,9283.38,inf


In [190]:
df_importation['%_evo_importation'] = df_importation['%_evo_importation'].replace([np.inf, -np.inf], np.nan)
df_importation['%_evo_importation'] = df_importation['%_evo_importation'].fillna(1000)

<div style="background-color: RGB(51,165,182);" >
<h2 style="margin: auto; padding: 20px; color:#fff; ">Merge des fichiers</h2>
</div>

In [191]:
# Dictionnaire complet d'harmonisation des pays
dict_harmonisation = {
    # Traductions Anglais -> Français (Pour df_politic, df_IDH, df_ELEC, df_LPI, df_TARIFS)
    'Afghanistan': 'Afghanistan', 'Albania': 'Albanie', 'Algeria': 'Algérie', 'Angola': 'Angola','Arabie saoudite': 'Arabie Saoudite',
    'Argentina': 'Argentine', 'Armenia': 'Arménie', 'Australia': 'Australie', 'Austria': 'Autriche', 
    'Azerbaijan': 'Azerbaïdjan', 'Bahrain': 'Bahreïn', 'Bangladesh': 'Bangladesh', 'Barbados': 'Barbade', 
    'Belarus': 'Biélorussie','Bélarus': 'Biélorussie', 'Belgium': 'Belgique', 'Benin': 'Bénin', 'Bhutan': 'Bhoutan', 'Bolivia': 'Bolivie', 
    'Bosnia and Herzegovina': 'Bosnie-Herzégovine', 'Botswana': 'Botswana', 'Brazil': 'Brésil', 
    'Bulgaria': 'Bulgarie', 'Burkina Faso': 'Burkina Faso', 'Burundi': 'Burundi', 'Cambodia': 'Cambodge', 
    'Cameroon': 'Cameroun', 'Canada': 'Canada', 'Cape Verde': 'Cabo Verde', 'Central African Republic': 'République centrafricaine', 
    'Chad': 'Tchad', 'Chile': 'Chili', 'China': 'Chine', 'Colombia': 'Colombie', 'Comoros': 'Comores', 
    'Congo': 'Congo', 'Costa Rica': 'Costa Rica', "Cote d'Ivoire": "Côte d'Ivoire", 'Croatia': 'Croatie', 
    'Cuba': 'Cuba', 'Cyprus': 'Chypre', 'Czechia': 'Tchéquie', 'Democratic Republic of Congo': 'République démocratique du Congo', 
    'Congo, Dem. Rep.': 'République démocratique du Congo', 'Congo, Rep.': 'Congo', 'Denmark': 'Danemark', 
    'Djibouti': 'Djibouti', 'Dominican Republic': 'République dominicaine', 'East Timor': 'Timor-Leste', 
    'Ecuador': 'Équateur', 'Egypt': 'Égypte', 'Egypt, Arab Rep.': 'Égypte', 'El Salvador': 'El Salvador', 
    'Equatorial Guinea': 'Guinée équatoriale', 'Eritrea': 'Érythrée', 'Estonia': 'Estonie', 'Eswatini': 'Eswatini', 
    'Ethiopia': 'Éthiopie', 'Fiji': 'Fidji', 'Finland': 'Finlande', 'France': 'France', 'Gabon': 'Gabon', 
    'Gambia': 'Gambie', 'Gambia, The': 'Gambie', 'Georgia': 'Géorgie', 'Germany': 'Allemagne', 'Ghana': 'Ghana', 
    'Greece': 'Grèce', 'Guatemala': 'Guatemala', 'Guinea': 'Guinée', 'Guinea-Bissau': 'Guinée-Bissau', 
    'Guyana': 'Guyana', 'Haiti': 'Haïti', 'Honduras': 'Honduras', 'Hong Kong': 'Chine - RAS de Hong-Kong', 
    'Hong Kong SAR, China': 'Chine - RAS de Hong-Kong', 'Hungary': 'Hongrie', 'Iceland': 'Islande', 
    'India': 'Inde', 'Indonesia': 'Indonésie', 'Iran': "Iran (République islamique d')", 'Iran, Islamic Rep.': "Iran (République islamique d')", 
    'Iraq': 'Iraq', 'Ireland': 'Irlande', 'Israel': 'Israël', 'Italy': 'Italie', 'Jamaica': 'Jamaïque', 
    'Japan': 'Japon', 'Jordan': 'Jordanie', 'Kazakhstan': 'Kazakhstan', 'Kenya': 'Kenya', 'Kuwait': 'Koweït', 
    'Kyrgyzstan': 'Kirghizistan', 'Kyrgyz Republic': 'Kirghizistan', 'Laos': 'République démocratique populaire lao', 
    'Lao PDR': 'République démocratique populaire lao', 'Latvia': 'Lettonie', 'Lebanon': 'Liban', 'Lesotho': 'Lesotho', 
    'Liberia': 'Libéria', 'Libya': 'Libye', 'Lithuania': 'Lituanie', 'Luxembourg': 'Luxembourg', 'Madagascar': 'Madagascar', 
    'Malawi': 'Malawi', 'Malaysia': 'Malaisie', 'Maldives': 'Maldives', 'Mali': 'Mali', 'Malta': 'Malte', 
    'Mauritania': 'Mauritanie', 'Mauritius': 'Maurice', 'Mexico': 'Mexique', 'Moldova': 'République de Moldova', 
    'Mongolia': 'Mongolie', 'Montenegro': 'Monténegro', 'Morocco': 'Maroc', 'Mozambique': 'Mozambique', 
    'Myanmar': 'Myanmar', 'Namibia': 'Namibie', 'Nepal': 'Népal', 'Netherlands': 'Pays-Bas (Royaume des)', 
    'New Zealand': 'Nouvelle-Zélande', 'Nicaragua': 'Nicaragua', 'Niger': 'Niger', 'Nigeria': 'Nigéria', 
    'North Korea': 'République populaire démocratique de Corée', "Korea, Dem. People's Rep.": 'République populaire démocratique de Corée', 
    'North Macedonia': 'Macédoine du Nord', 'Norway': 'Norvège', 'Oman': 'Oman', 'Pakistan': 'Pakistan', 
    'Panama': 'Panama', 'Papua New Guinea': 'Papouasie-Nouvelle-Guinée', 'Paraguay': 'Paraguay', 'Peru': 'Pérou', 
    'Philippines': 'Philippines', 'Poland': 'Pologne', 'Portugal': 'Portugal', 'Qatar': 'Qatar', 'Romania': 'Roumanie', 
    'Russia': 'Fédération de Russie', 'Russian Federation': 'Fédération de Russie', 'Rwanda': 'Rwanda', 
    'Saudi Arabia': 'Arabie Saoudite', 'Senegal': 'Sénégal', 'Serbia': 'Serbie', 'Seychelles': 'Seychelles', 
    'Sierra Leone': 'Sierra Leone', 'Singapore': 'Singapour', 'Slovakia': 'Slovaquie', 'Slovak Republic': 'Slovaquie', 
    'Slovenia': 'Slovénie', 'Solomon Islands': 'Îles Salomon', 'Somalia': 'Somalie', 'South Africa': 'Afrique du Sud', 
    'South Korea': 'République de Corée', 'Korea, Rep.': 'République de Corée', 'South Sudan': 'Soudan du Sud', 
    'Spain': 'Espagne', 'Sri Lanka': 'Sri Lanka', 'Sudan': 'Soudan', 'Suriname': 'Suriname', 'Sweden': 'Suède', 
    'Switzerland': 'Suisse', 'Syria': 'République arabe syrienne', 'Syrian Arab Republic': 'République arabe syrienne', 
    'Taiwan': 'Chine, Taiwan Province de', 'Tajikistan': 'Tadjikistan', 'Tanzania': 'République-Unie de Tanzanie', 
    'Thailand': 'Thaïlande', 'Togo': 'Togo', 'Trinidad and Tobago': 'Trinité-et-Tobago', 'Tunisia': 'Tunisie', 
    'Turkey': 'Turquie', 'Turkiye': 'Turquie', 'Turkmenistan': 'Turkménistan', 'Uganda': 'Ouganda', 'Ukraine': 'Ukraine', 
    'United Arab Emirates': 'Émirats arabes unis', 'United Kingdom': "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord", 
    'United States': "États-Unis d'Amérique", 'Uruguay': 'Uruguay', 'Uzbekistan': 'Ouzbékistan', 'Vanuatu': 'Vanuatu', 
    'Venezuela': 'Venezuela (République bolivarienne du)', 'Venezuela, RB': 'Venezuela (République bolivarienne du)', 
    'Vietnam': 'Viet Nam', 'Viet Nam': 'Viet Nam', 'Yemen': 'Yémen', 'Yemen, Rep.': 'Yémen', 'Zambia': 'Zambie', 'Zimbabwe': 'Zimbabwe',
    
    # Harmonisations pour les fichiers FAO (Français vers format uniforme court si tu veux, ou l'inverse)
    'Bolivie (État plurinational de)': 'Bolivie',
    'Chine, continentale': 'Chine'
}

In [192]:
liste_regions_a_exclure = [
    'World', 'World (population-weighted)', 'Europe', 'Europe (population-weighted)', 'Asia', 'Asia (population-weighted)',
    'Africa', 'Africa (population-weighted)', 'North America', 'North America (population-weighted)', 
    'South America', 'South America (population-weighted)', 'Oceania', 'Oceania (population-weighted)',
    'European Union (27)', 'European Union', 'Euro area', 'High-income countries', 'High income',
    'Low-income countries', 'Low income', 'Lower-middle-income countries', 'Lower middle income',
    'Upper-middle-income countries', 'Upper middle income', 'Low & middle income', 'Middle income',
    'Africa Eastern and Southern', 'Africa Western and Central', 'Arab World', 'Arab States (UNDP)',
    'Caribbean small states', 'Central Europe and the Baltics', 'Early-demographic dividend',
    'East Asia & Pacific', 'East Asia & Pacific (IDA & IBRD countries)', 'East Asia & Pacific (excluding high income)',
    'Europe & Central Asia', 'Europe & Central Asia (IDA & IBRD countries)', 'Europe & Central Asia (excluding high income)',
    'Fragile and conflict affected situations', 'Heavily indebted poor countries (HIPC)', 'IBRD only',
    'IDA & IBRD total', 'IDA blend', 'IDA only', 'IDA total', 'Late-demographic dividend',
    'Latin America & Caribbean', 'Latin America & Caribbean (excluding high income)', 
    'Latin America & the Caribbean (IDA & IBRD countries)', 'Least developed countries: UN classification',
    'Middle East, North Africa, Afghanistan & Pakistan', 'OECD members', 'Pacific island small states',
    'Post-demographic dividend', 'Pre-demographic dividend', 'Sub-Saharan Africa', 
    'Sub-Saharan Africa (IDA & IBRD countries)', 'Sub-Saharan Africa (excluding high income)',
    'South Asia', 'South Asia (IDA & IBRD)', 'South Asia (UNDP)', 'Very high human development (UNDP)',
    'High human development (UNDP)', 'Medium human development (UNDP)', 'Low human development (UNDP)',
    'East Asia and the Pacific (UNDP)', 'Europe and Central Asia (UNDP)', 'Latin America and the Caribbean (UNDP)',
    'Sub-Saharan Africa (UNDP)', 'Small states', 'Other small states', 'Not classified'
]

In [193]:
# 1. On regroupe tes DataFrames dans un dictionnaire simple (Clé = Nom du DataFrame, Valeur = Le DataFrame)
mes_dfs = {
    'df_population': df_population,
    'df_idh': df_idh,
    'df_pop_alim_saine': df_pop_alim_saine,
    'df_cout_alim_saine': df_cout_alim_saine,
    'df_pib': df_pib,
    'df_importation': df_importation,
    'df_politic': df_politic,
    'df_tarifs': df_tarifs,
    'df_elec': df_elec,
    'df_lpi': df_lpi,
    'df_consommation_poulet' : df_consommation_poulet
}

# 2. On applique les transformations (Traduction + Filtrage des régions)
for nom_df, df in mes_dfs.items():
    # Traduction avec le dictionnaire d'harmonisation
    df['Zone'] = df['Zone'].replace(dict_harmonisation)
    
    # Filtrage des lignes (on garde uniquement ce qui n'est pas dans la liste à exclure)
    # On réassigne directement dans le dictionnaire pour que le changement soit enregistré
    mes_dfs[nom_df] = df[~df['Zone'].isin(liste_regions_a_exclure)]

# 3. Facultatif : Si tu as besoin de retrouver tes variables d'origine nettoyées
df_population = mes_dfs['df_population']
df_idh = mes_dfs['df_idh']
df_pop_alim_saine = mes_dfs['df_pop_alim_saine']
df_cout_alim_saine = mes_dfs['df_cout_alim_saine']
df_pib = mes_dfs['df_pib']
df_importation = mes_dfs['df_importation']
df_politic = mes_dfs['df_politic']
df_tarifs = mes_dfs['df_tarifs']
df_elec = mes_dfs['df_elec']
df_lpi = mes_dfs['df_lpi']
df_consommation_poulet = mes_dfs['df_consommation_poulet']

print("Vos DataFrames sont désormais uniformisés en français et nettoyés des zones globales ! Ready pour le merge.")

Vos DataFrames sont désormais uniformisés en français et nettoyés des zones globales ! Ready pour le merge.


In [194]:
print("--- PAYS DF POPULATION ---")
print(sorted(list(df_population['Zone'].unique())))

print("\n--- PAYS DF POLITIC ---")
print(sorted(list(df_politic['Zone'].unique())))

print("\n--- PAYS DF Dist ---")
print(sorted(list(df_dist['Code'].unique())))

print("\n--- PAYS DF IDH ---")
print(sorted(list(df_idh['Zone'].unique())))

print("\n--- PAYS DF POP_ALIM_SAINE ---")
print(sorted(list(df_pop_alim_saine['Zone'].unique())))

print("\n--- PAYS DF COUT_ALIM_SAine ---")
print(sorted(list(df_cout_alim_saine['Zone'].unique())))

print("\n--- PAYS DF ELEC ---")
print(sorted(list(df_elec['Zone'].unique())))

print("\n--- PAYS DF LPI ---")
print(sorted(list(df_lpi['Zone'].unique())))

print("\n--- PAYS DF PIB ---")
print(sorted(list(df_pib['Zone'].unique())))

print("\n--- PAYS DF TARIFS ---")
print(sorted(list(df_tarifs['Zone'].unique())))

print("\n--- PAYS DF Import ---")
print(sorted(list(df_importation['Zone'].unique())))

print("\n--- PAYS DF CONSO ---")
print(sorted(list(df_consommation_poulet['Zone'].unique())))


--- PAYS DF POPULATION ---
['Afghanistan', 'Afrique du Sud', 'Albanie', 'Algérie', 'Allemagne', 'Andorre', 'Angola', 'Anguilla', 'Antigua-et-Barbuda', 'Arabie Saoudite', 'Argentine', 'Arménie', 'Aruba', 'Ascension, Sainte-Hélène et Tristan da Cunha', 'Australie', 'Autriche', 'Azerbaïdjan', 'Bahamas', 'Bahreïn', 'Bangladesh', 'Barbade', 'Belgique', 'Belize', 'Bermudes', 'Bhoutan', 'Biélorussie', 'Bolivie', 'Bonaire, Saint-Eustache et Saba', 'Bosnie-Herzégovine', 'Botswana', 'Brunéi Darussalam', 'Brésil', 'Bulgarie', 'Burkina Faso', 'Burundi', 'Bénin', 'Cabo Verde', 'Cambodge', 'Cameroun', 'Canada', 'Chili', 'Chine', 'Chine - RAS de Hong-Kong', 'Chine - RAS de Macao', 'Chine, Taiwan Province de', 'Chypre', 'Colombie', 'Comores', 'Congo', 'Costa Rica', 'Croatie', 'Cuba', 'Curaçao', "Côte d'Ivoire", 'Danemark', 'Djibouti', 'Dominique', 'El Salvador', 'Espagne', 'Estonie', 'Eswatini', 'Fidji', 'Finlande', 'France', 'Fédération de Russie', 'Gabon', 'Gambie', 'Ghana', 'Gibraltar', 'Grenade', 

In [195]:

# 1. Dictionnaire de conversion pour le fichier des Distances (ISO3 -> Français)
# Complète ce dictionnaire au besoin selon tes besoins géographiques
dict_iso_vers_fr = {
    'AFG': 'Afghanistan', 'AGO': 'Angola', 'ALB': 'Albanie', 'AND': 'Andorre', 
    'ARE': 'Émirats arabes unis', 'ARG': 'Argentine', 'ARM': 'Arménie', 'ATG': 'Antigua-et-Barbuda', 
    'AUS': 'Australie', 'AUT': 'Autriche', 'AZE': 'Azerbaïdjan', 'BDI': 'Burundi', 
    'BEL': 'Belgique', 'BEN': 'Bénin', 'BFA': 'Burkina Faso', 'BGD': 'Bangladesh', 
    'BGR': 'Bulgarie', 'BHR': 'Bahreïn', 'BHS': 'Bahamas', 'BIH': 'Bosnie-Herzégovine', 
    'BLR': 'Biélorussie', 'BLZ': 'Belize', 'BOL': 'Bolivie', 'BRA': 'Brésil', 
    'BRB': 'Barbade', 'BRN': 'Brunéi Darussalam', 'BTN': 'Bhoutan', 'BWA': 'Botswana', 
    'CAF': 'République centrafricaine', 'CAN': 'Canada', 'CHE': 'Suisse', 'CHL': 'Chili', 
    'CHN': 'Chine', 'CIV': "Côte d'Ivoire", 'CMR': 'Cameroun', 'COG': 'Congo', 
    'COL': 'Colombie', 'COM': 'Comores', 'CPV': 'Cabo Verde', 'CRI': 'Costa Rica', 
    'CUB': 'Cuba', 'CYP': 'Chypre', 'CZE': 'Tchéquie', 'DEU': 'Allemagne', 
    'DJI': 'Djibouti', 'DMA': 'Dominique', 'DNK': 'Danemark', 'DOM': 'République dominicaine', 
    'DZA': 'Algérie', 'ECU': 'Équateur', 'EGY': 'Égypte', 'ERI': 'Érythrée', 
    'ESP': 'Espagne', 'EST': 'Estonie', 'ETH': 'Éthiopie', 'FIN': 'Finlande', 
    'FJI': 'Fidji', 'FRA': 'France', 'GAB': 'Gabon', 'GBR': "Royaume-Uni de Grande-Bretagne et d'Irlande du Nord", 
    'GEO': 'Géorgie', 'GHA': 'Ghana', 'GIN': 'Guinée', 'GMB': 'Gambie', 
    'GNB': 'Guinée-Bissau', 'GNQ': 'Guinée équatoriale', 'GRC': 'Grèce', 'GRD': 'Grenade', 
    'GRL': 'Groenland', 'GTM': 'Guatemala', 'GUY': 'Guyana', 'HND': 'Honduras', 
    'HRV': 'Croatie', 'HTI': 'Haïti', 'HUN': 'Hongrie', 'IDN': 'Indonésie', 
    'IND': 'Inde', 'IRL': 'Irlande', 'IRN': "Iran (République islamique d')", 
    'IRQ': 'Iraq', 'ISL': 'Islande', 'ISR': 'Israël', 'ITA': 'Italie', 
    'JAM': 'Jamaïque', 'JOR': 'Jordanie', 'JPN': 'Japon', 'KAZ': 'Kazakhstan', 
    'KEN': 'Kenya', 'KGZ': 'Kirghizistan', 'KHM': 'Cambodge', 'KIR': 'Kiribati', 
    'KNA': 'Saint-Kitts-et-Nevis', 'KOR': 'République de Corée', 'KWT': 'Koweït', 
    'LAO': 'République démocratique populaire lao', 'LBN': 'Liban', 'LBR': 'Libéria', 
    'LBY': 'Libye', 'LCA': 'Sainte-Lucie', 'LKA': 'Sri Lanka', 'LSO': 'Lesotho', 
    'LTU': 'Lituanie', 'LUX': 'Luxembourg', 'LVA': 'Lettonie', 'MAR': 'Maroc', 
    'MDA': 'République de Moldova', 'MDG': 'Madagascar', 'MDV': 'Maldives', 
    'MEX': 'Mexique', 'MKD': 'Macédoine du Nord', 'MLI': 'Mali', 'MLT': 'Malte', 
    'MMR': 'Myanmar', 'MNG': 'Mongolie', 'MOZ': 'Mozambique', 'MRT': 'Mauritanie', 
    'MUS': 'Maurice', 'MWI': 'Malawi', 'MYS': 'Malaisie', 'NAM': 'Namibie', 
    'NER': 'Niger', 'NGA': 'Nigéria', 'NIC': 'Nicaragua', 'NLD': 'Pays-Bas (Royaume des)', 
    'NOR': 'Norvège', 'NPL': 'Népal', 'NRU': 'Nauru', 'NZL': 'Nouvelle-Zélande', 
    'OMN': 'Oman', 'PAK': 'Pakistan', 'PAN': 'Panama', 'PER': 'Pérou', 
    'PHL': 'Philippines', 'PLW': 'Palaos', 'PNG': 'Papouasie-Nouvelle-Guinée', 
    'POL': 'Pologne', 'PRK': 'République populaire démocratique de Corée', 
    'PRT': 'Portugal', 'PRY': 'Paraguay', 'QAT': 'Qatar', 'ROU': 'Roumanie', 
    'RUS': 'Fédération de Russie', 'RWA': 'Rwanda', 'SAU': 'Arabie Saoudite', 
    'SDN': 'Soudan', 'SEN': 'Sénégal', 'SGP': 'Singapour', 'SLB': 'Îles Salomon', 
    'SLE': 'Sierra Leone', 'SLV': 'El Salvador', 'SMR': 'Saint-Marin', 'SOM': 'Somalie', 
    'STP': 'Sao Tomé-et-Principe', 'SUR': 'Suriname', 'SVK': 'Slovaquie', 
    'SVN': 'Slovénie', 'SWE': 'Suède', 'SWZ': 'Eswatini', 'SYC': 'Seychelles', 
    'SYR': 'République arabe syrienne', 'TCD': 'Tchad', 'TGO': 'Togo', 
    'THA': 'Thaïlande', 'TJK': 'Tadjikistan', 'TKM': 'Turkménistan', 'TLS': 'Timor-Leste', 
    'TON': 'Tonga', 'TTO': 'Trinité-et-Tobago', 'TUN': 'Tunisie', 'TUR': 'Türkiye', 
    'TUV': 'Tuvalu', 'TWN': 'Chine, Taiwan Province de', 'TZA': 'République-Unie de Tanzanie', 
    'UGA': 'Ouganda', 'UKR': 'Ukraine', 'URY': 'Uruguay', 'USA': "États-Unis d'Amérique", 
    'UZB': 'Ouzbékistan', 'VCT': 'Saint-Vincent-et-les Grenadines', 'VEN': 'Venezuela (République bolivarienne du)', 
    'VNM': 'Viet Nam', 'VUT': 'Vanuatu', 'WSM': 'Samoa', 'YEM': 'Yémen', 
    'ZAF': 'Afrique du Sud', 'ZMB': 'Zambie', 'ZWE': 'Zimbabwe'
}

# 2. Dictionnaire global d'harmonisation Anglais / Formats longs -> Français standard
dict_traduction_pays = {
    'Andorra': 'Andorre', 'Antigua and Barbuda': 'Antigua-et-Barbuda', 'Brunei': 'Brunéi Darussalam',
    'Dominica': 'Dominique', 'Grenada': 'Grenade', 'Marshall Islands': 'Îles Marshall',
    'Micronesia (country)': 'Micronésie (États fédérés de)', 'Palau': 'Palaos',
    'Saint Kitts and Nevis': 'Saint-Kitts-et-Nevis', 'Saint Lucia': 'Sainte-Lucie',
    'Saint Vincent and the Grenadines': 'Saint-Vincent-et-les Grenadines', 'San Marino': 'Saint-Marin',
    'Sao Tome and Principe': 'Sao Tomé-et-Principe', 'Bahamas, The': 'Bahamas',
    'Bermuda': 'Bermudes', 'British Virgin Islands': 'Îles Vierges britanniques',
    'Brunei Darussalam': 'Brunéi Darussalam', 'Cayman Islands': 'Îles Caïmanes',
    'Curacao': 'Curaçao', 'Greenland': 'Groenland', 'Macao SAR, China': 'Chine - RAS de Macao',
    'Micronesia, Fed. Sts.': 'Micronésie (États fédérés de)', 'New Caledonia': 'Nouvelle-Calédonie',
    'Northern Mariana Islands': 'Îles Mariannes du Nord', 'Puerto Rico (US)': 'Porto Rico',
    'Sint Maarten (Dutch part)': 'Sint Maarten (partie néerlandaise)', 'Somalia, Fed. Rep.': 'Somalie',
    'St. Kitts and Nevis': 'Saint-Kitts-et-Nevis', 'St. Lucia': 'Sainte-Lucie',
    'St. Martin (French part)': 'Saint-Martin (partie française)',
    'St. Vincent and the Grenadines': 'Saint-Vincent-et-les Grenadines',
    'Turks and Caicos Islands': 'Îles Turques-et-Caïques', 'Virgin Islands (U.S.)': 'Îles Vierges américaines',
    'Bolivie (État plurinational de)': 'Bolivie', 'Chine, continentale': 'Chine'
}

# --- APPLICATION DES CORRECTIONS AVANT MERGE ---

# On traduit en premier le df des distances de ses codes ISO3 vers les noms français
df_dist['Zone'] = df_dist['Code'].replace(dict_iso_vers_fr)

# On prépare la base principale
df_final = df_population.copy()

# Liste finale ordonnée de tous les autres indicateurs à fusionner
autres_dfs = [
    df_idh, df_pop_alim_saine, df_cout_alim_saine, 
    df_pib, df_importation, df_politic, df_tarifs, df_elec, df_lpi, df_consommation_poulet, df_dist
]

for i, df in enumerate(autres_dfs):
    df_temp = df.copy()
    
    # Traduction systématique des anomalies textuelles / anglais
    df_temp['Zone'] = df_temp['Zone'].replace(dict_traduction_pays)
    
    # Nettoyage des agrégats régionaux mondiaux propres à ce df
    df_temp = df_temp[~df_temp['Zone'].isin(liste_regions_a_exclure)]
    
    # Nettoyage des colonnes doublons
    colonnes_doublons = [col for col in df_temp.columns if col in df_final.columns and col != 'Zone']
    df_temp = df_temp.drop(columns=colonnes_doublons)
    
    # Fusion
    df_final = pd.merge(df_final, df_temp, on='Zone', how='left')

# Nettoyage final du DataFrame agrégé
df_final = df_final[~df_final['Zone'].isin(liste_regions_a_exclure)]

# Relance la vérification du taux de NaN
print("Nouveau taux de NaN par indicateur :")
print(df_final.isnull().mean() * 100)

Nouveau taux de NaN par indicateur :
Zone                                0.000000
Année                               0.000000
population                          0.000000
evo_pop                             0.000000
Code                               17.500000
Human Development Index            17.500000
%_evo_idh                          17.916667
pop_unable_to_eat_healthy          24.166667
%_evo_pop_unable_to_eat_healthy    24.166667
cout_alim_saine                    27.916667
PIB                                 9.583333
%_evo_pib                           9.583333
importation                        25.000000
%_evo_importation                  25.000000
Political polarization score       25.416667
%_evo_politic                      25.416667
tarifs                             38.750000
%_evo_tarifs                       38.750000
Access_Elec_Pct                    10.416667
%_evo_elec                         10.416667
LPI                                27.500000
%_evo_lpi         

In [196]:
df_population.loc[df_population['Zone'] == 'Arabie Saoudite']

,Zone,Année,population,evo_pop
800,Arabie Saoudite,2023,33264292.0,18.687336


In [197]:
df_final_2 = df_final.drop(columns=['tarifs', '%_evo_tarifs', 'Code', 'Année'])
print(df_final_2.isnull().mean() * 100)

Zone                                0.000000
population                          0.000000
evo_pop                             0.000000
Human Development Index            17.500000
%_evo_idh                          17.916667
pop_unable_to_eat_healthy          24.166667
%_evo_pop_unable_to_eat_healthy    24.166667
cout_alim_saine                    27.916667
PIB                                 9.583333
%_evo_pib                           9.583333
importation                        25.000000
%_evo_importation                  25.000000
Political polarization score       25.416667
%_evo_politic                      25.416667
Access_Elec_Pct                    10.416667
%_evo_elec                         10.416667
LPI                                27.500000
%_evo_lpi                          30.000000
consommation_poulet                23.750000
%_evo_consommation_poulet          27.083333
dist                               20.000000
dtype: float64


In [198]:
# Supprime toutes les lignes contenant au moins une valeur manquante (NaN)
df_final_2 = df_final_2.dropna()
df_final_2 = df_final_2.drop_duplicates()

# Vérification : tout doit être à 0% !
print(df_final_2.isnull().mean() * 100)
print(f"Nombre de pays restants : {df_final_2['Zone'].nunique()}")
df_final_2 = df_final_2.drop_duplicates()

# Vérification : tout doit être à 0% !
print(df_final_2.isnull().mean() * 100)
print(f"Nombre de pays restants : {df_final_2['Zone'].nunique()}")

Zone                               0.0
population                         0.0
evo_pop                            0.0
Human Development Index            0.0
%_evo_idh                          0.0
pop_unable_to_eat_healthy          0.0
%_evo_pop_unable_to_eat_healthy    0.0
cout_alim_saine                    0.0
PIB                                0.0
%_evo_pib                          0.0
importation                        0.0
%_evo_importation                  0.0
Political polarization score       0.0
%_evo_politic                      0.0
Access_Elec_Pct                    0.0
%_evo_elec                         0.0
LPI                                0.0
%_evo_lpi                          0.0
consommation_poulet                0.0
%_evo_consommation_poulet          0.0
dist                               0.0
dtype: float64
Nombre de pays restants : 123
Zone                               0.0
population                         0.0
evo_pop                            0.0
Human Development I

In [199]:
# Supprime toutes les lignes contenant au moins une valeur manquante (NaN)
df_sans_nan = df_final.dropna()
df_sans_nan = df_sans_nan.drop_duplicates()

# Vérification : tout doit être à 0% !
print(df_sans_nan.isnull().mean() * 100)
print(f"Nombre de pays restants : {df_sans_nan['Zone'].nunique()}")

Zone                               0.0
Année                              0.0
population                         0.0
evo_pop                            0.0
Code                               0.0
Human Development Index            0.0
%_evo_idh                          0.0
pop_unable_to_eat_healthy          0.0
%_evo_pop_unable_to_eat_healthy    0.0
cout_alim_saine                    0.0
PIB                                0.0
%_evo_pib                          0.0
importation                        0.0
%_evo_importation                  0.0
Political polarization score       0.0
%_evo_politic                      0.0
tarifs                             0.0
%_evo_tarifs                       0.0
Access_Elec_Pct                    0.0
%_evo_elec                         0.0
LPI                                0.0
%_evo_lpi                          0.0
consommation_poulet                0.0
%_evo_consommation_poulet          0.0
dist                               0.0
dtype: float64
Nombre de 

In [200]:
# 1. On repère la ligne de la Chine globale et on remplit ses quelques NaN par la médiane pour éviter qu'elle soit supprimée
df_final_2 = df_final_2.drop_duplicates(subset=["Zone"], keep="first")

# Vérification finale
print(f"Nombre de pays restants : {df_final_2['Zone'].nunique()}")
print("Est-ce que la Chine est bien présente ? :", 'Chine' in df_final_2['Zone'].values)

Nombre de pays restants : 123
Est-ce que la Chine est bien présente ? : True


In [201]:
df_sans_nan.to_csv('df_final_acp.csv', index=False)

In [202]:
# 1. On repère la ligne de la Chine globale et on remplit ses quelques NaN par la médiane pour éviter qu'elle soit supprimée
df_final_2 = df_final_2.drop_duplicates(subset=["Zone"], keep="first")
df_final_2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 123 entries, 1 to 238
Data columns (total 21 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   Zone                             123 non-null    object 
 1   population                       123 non-null    float64
 2   evo_pop                          123 non-null    float64
 3   Human Development Index          123 non-null    float64
 4   %_evo_idh                        123 non-null    float64
 5   pop_unable_to_eat_healthy        123 non-null    float64
 6   %_evo_pop_unable_to_eat_healthy  123 non-null    float64
 7   cout_alim_saine                  123 non-null    float64
 8   PIB                              123 non-null    float64
 9   %_evo_pib                        123 non-null    float64
 10  importation                      123 non-null    float64
 11  %_evo_importation                123 non-null    float64
 12  Political polarization scor

In [203]:
df_final_2['Zone'].unique()

array(['Afrique du Sud', 'Albanie', 'Algérie', 'Allemagne', 'Angola',
       'Arabie Saoudite', 'Arménie', 'Australie', 'Autriche',
       'Azerbaïdjan', 'Bangladesh', 'Biélorussie', 'Belgique', 'Bolivie',
       'Bosnie-Herzégovine', 'Botswana', 'Brésil', 'Bulgarie',
       'Burkina Faso', 'Cambodge', 'Cameroun', 'Canada', 'Chili', 'Chine',
       'Chypre', 'Colombie', 'Comores', 'Congo', 'Costa Rica',
       "Côte d'Ivoire", 'Croatie', 'Danemark', 'Djibouti', 'Égypte',
       'Émirats arabes unis', 'Équateur', 'Espagne', 'Estonie',
       "États-Unis d'Amérique", 'Éthiopie', 'Fédération de Russie',
       'Fidji', 'Finlande', 'France', 'Gabon', 'Gambie', 'Ghana', 'Grèce',
       'Guatemala', 'Guinée', 'Guinée-Bissau', 'Honduras', 'Hongrie',
       'Inde', 'Indonésie', 'Iraq', 'Irlande', 'Islande', 'Israël',
       'Italie', 'Jamaïque', 'Jordanie', 'Kazakhstan', 'Kenya',
       'Kirghizistan', 'Koweït', 'Lesotho', 'Lettonie', 'Liban',
       'Libéria', 'Lituanie', 'Luxembourg', 'Macéd

In [204]:
df_corresp = pd.read_csv("curiexplore-pays.csv", sep=";")
df_corresp = df_corresp[['iso3', 'name_fr']]
df_corresp = df_corresp.rename(columns={'name_fr' : 'Zone'})

df_final_2 = pd.merge(df_final_2, df_corresp, on='Zone', how='left')

In [205]:
import pycountry_convert as pc

def country_to_continent(country_code_iso3):
    try:
        country_alpha2 = pc.country_alpha3_to_country_alpha2(country_code_iso3)
        country_continent_code = pc.country_alpha2_to_continent_code(country_alpha2)
        country_continent_name = pc.convert_continent_code_to_continent_name(country_continent_code)
        return country_continent_name
    except:
        return "Inconnu"

# 3. Application sur la colonne iso3
df_final_2['Continent'] = df_final_2['iso3'].apply(country_to_continent)

# 4. Vérification
print(df_final_2[['Zone', 'iso3', 'Continent']].head(20))

                  Zone iso3      Continent
0       Afrique du Sud  ZAF         Africa
1              Albanie  ALB         Europe
2              Algérie  DZA         Africa
3            Allemagne  DEU         Europe
4               Angola  AGO         Africa
5      Arabie Saoudite  SAU           Asia
6              Arménie  ARM           Asia
7            Australie  AUS        Oceania
8             Autriche  AUT         Europe
9          Azerbaïdjan  AZE           Asia
10          Bangladesh  BGD           Asia
11         Biélorussie  BLR         Europe
12            Belgique  BEL         Europe
13             Bolivie  BOL  South America
14  Bosnie-Herzégovine  BIH         Europe
15            Botswana  BWA         Africa
16              Brésil  BRA  South America
17            Bulgarie  BGR         Europe
18        Burkina Faso  BFA         Africa
19            Cambodge  KHM           Asia


In [209]:
df_final_2.loc[df_final_2['Continent'] == 'Inconnu', 'Zone']

Series([], Name: Zone, dtype: object)

In [207]:
df_final_2.to_csv('df_final_2_acp.csv', index=False)